# 04c - Recomendador avanzado

Este notebook mejora los recomendadores anteriores usando solo datos ya disponibles en el proyecto: generos, tags, valoracion media y numero de valoraciones.

La idea es combinar varias señales para obtener recomendaciones mas utiles y mas robustas que las basadas solo en similitud de contenido.

## 1. Por que esta version mejora las anteriores

- Los generos ofrecen una descripcion general, pero los tags aportan matices mas finos sobre tema, tono o estilo.
- La valoracion media ayuda a favorecer peliculas mejor recibidas.
- El numero de valoraciones reduce el riesgo de priorizar peliculas con pocos votos.
- La valoracion ponderada corrige el sesgo de peliculas con muy pocas valoraciones.
- La popularidad suavizada ayuda a dar mas peso a opciones mas fiables sin depender solo de la similitud.
- Los filtros por ano y generos permiten adaptar la busqueda a cada caso.

Con esto, el recomendador no solo busca peliculas parecidas, sino tambien mas solidas desde el punto de vista de la confianza en los datos.

## 2. Importacion de librerias

In [50]:
from pathlib import Path
import pathlib
import re
import sys
import time
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from deep_translator import GoogleTranslator
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

base_dir = Path.cwd().resolve()
if not (base_dir / 'src').exists():
    if (base_dir / 'notebooks').exists():
        base_dir = base_dir
    elif (base_dir.parent / 'src').exists():
        base_dir = base_dir.parent
    else:
        candidate = base_dir
        for parent in base_dir.parents:
            if (parent / 'src').exists():
                candidate = parent
                break
        base_dir = candidate

sys.path.insert(0, str(base_dir))

from src.recommender import (
    build_genre_feature_matrix,
    find_movie_index,
    get_genre_columns,
    recommend_movies_by_genres,
)
movies_path = base_dir / 'data' / 'processed' / 'movies_clean.csv'
tags_path = base_dir / 'data' / 'raw' / 'tags.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 90)

## 3. Carga de datos y comprobacion de archivos

Antes de leer los CSV, comprobamos que existan. Si falta alguno, el notebook se detiene con un error claro.

In [51]:
required_files = {
    'movies_clean': movies_path,
    'tags': tags_path,
}

missing_files = [name for name, path in required_files.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'No se encontraron los archivos necesarios: {missing_files}')

movies_clean = pd.read_csv(movies_path)
tags_df = pd.read_csv(tags_path)

print('movies_clean shape:', movies_clean.shape)
print('tags_df shape:', tags_df.shape)
display(movies_clean.head())
display(tags_df.head())

movies_clean shape: (87585, 26)
tags_df shape: (2000072, 4)


,movieId,title,genres,year,title_clean,rating_mean,rating_count,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,3.897438,68997,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,3.275758,28904,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,3.139447,13134,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,2.845331,2806,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,3.059602,13154,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


## 4. Limpieza, normalizacion, traduccion y union de tags

Las etiquetas proceden de usuarios y pueden contener ruido, errores de encoding, mayusculas inconsistentes o incluso palabras en otros idiomas.

En esta fase se normalizan, se filtran las menos frecuentes, se traducen al ingles con cache local y, si una traduccion falla, se conserva el tag limpio original.

El objetivo es construir una base estable y reproducible antes de combinar los tags con `movies_clean`.

In [52]:
MIN_TAG_FREQUENCY = 2
TRANSLATION_SLEEP = 0
SAVE_EVERY = 10
translation_cache_path = base_dir / 'data' / 'processed' / 'tag_translation_cache.csv'
translation_cache_path.parent.mkdir(parents=True, exist_ok=True)


def normalize_text(text):
    if pd.isna(text):
        return ''

    text = str(text).strip().lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(character for character in text if not unicodedata.combining(character))
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = re.sub(r'[^a-z0-9\s-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


BLOCKED_TAGS = {'clv', 'grun running', 'no fa ganes'}


def is_useful_tag(tag):
    cleaned = normalize_text(tag)
    if not cleaned:
        return False
    if cleaned in BLOCKED_TAGS:
        return False
    if not re.search(r'[a-z]', cleaned):
        return False
    if re.fullmatch(r'[\d\s\-/]+', cleaned):
        return False

    letters = len(re.findall(r'[a-z]', cleaned))
    digits = len(re.findall(r'\d', cleaned))
    if digits > letters:
        return False
    if ' ' not in cleaned and len(cleaned) <= 3:
        return False

    return True


tags_clean = tags_df[['movieId', 'tag']].copy()
original_tag_count = int(tags_clean['tag'].notna().sum())

tags_clean = tags_clean.dropna(subset=['tag']).copy()
tags_clean['tag_clean'] = tags_clean['tag'].map(normalize_text)
tags_after_normalize = int(tags_clean['tag_clean'].str.len().gt(0).sum())
tags_clean = tags_clean[tags_clean['tag_clean'].map(is_useful_tag)].copy()
tags_after_useful = len(tags_clean)
tags_clean = tags_clean[tags_clean['tag_clean'].str.len() >= 3].copy()
tags_clean = tags_clean.drop_duplicates(subset=['movieId', 'tag_clean']).reset_index(drop=True)

tag_frequency = tags_clean['tag_clean'].value_counts()
tags_clean = tags_clean[tags_clean['tag_clean'].map(tag_frequency) >= MIN_TAG_FREQUENCY].copy().reset_index(drop=True)
tags_after_frequency = len(tags_clean)

unique_tags = tags_clean['tag_clean'].value_counts().index.tolist()
unique_valid_tags = len(unique_tags)

if translation_cache_path.exists() and translation_cache_path.stat().st_size > 0:
    translation_cache = pd.read_csv(translation_cache_path)
else:
    translation_cache = pd.DataFrame(columns=['tag_clean', 'tag_translated', 'translation_changed'])
    translation_cache.to_csv(translation_cache_path, index=False, encoding='utf-8')

if not {'tag_clean', 'tag_translated'}.issubset(translation_cache.columns):
    translation_cache = pd.DataFrame(columns=['tag_clean', 'tag_translated', 'translation_changed'])
else:
    if 'translation_changed' not in translation_cache.columns:
        translation_cache['translation_changed'] = False
    translation_cache = translation_cache[['tag_clean', 'tag_translated', 'translation_changed']].copy()

translation_cache['tag_clean'] = translation_cache['tag_clean'].map(normalize_text)
translation_cache['tag_translated'] = translation_cache['tag_translated'].fillna('').map(normalize_text)
translation_cache['translation_changed'] = translation_cache['tag_clean'] != translation_cache['tag_translated']
translation_cache = translation_cache[
    (translation_cache['tag_clean'] != '')
    & (translation_cache['tag_translated'] != '')
    & (translation_cache['tag_clean'].map(is_useful_tag))
    & (translation_cache['tag_clean'].isin(unique_tags))
].copy()
translation_cache = translation_cache.drop_duplicates(subset=['tag_clean'], keep='last').reset_index(drop=True)
translation_cache.to_csv(translation_cache_path, index=False, encoding='utf-8')

cached_tags = set(translation_cache['tag_clean'])
tags_to_translate = [tag for tag in unique_tags if tag not in cached_tags]

print('Numero total de tags unicos:', len(unique_tags))
print('Numero de tags ya en cache:', len(cached_tags))
print('Numero de tags pendientes de traducir:', len(tags_to_translate))

translator = GoogleTranslator(source='auto', target='en')
pending_rows = []

for index, tag in enumerate(tags_to_translate, start=1):
    try:
        translated = translator.translate(tag)
    except Exception:
        translated = tag

    translated_clean = normalize_text(translated)
    if not translated_clean:
        translated_clean = tag

    pending_rows.append({
        'tag_clean': tag,
        'tag_translated': translated_clean,
        'translation_changed': tag != translated_clean,
    })
    time.sleep(TRANSLATION_SLEEP)

    if index % SAVE_EVERY == 0 or index == len(tags_to_translate):
        if pending_rows:
            translation_cache = pd.concat([translation_cache, pd.DataFrame(pending_rows)], ignore_index=True)
            translation_cache = translation_cache.drop_duplicates(subset=['tag_clean'], keep='last').reset_index(drop=True)
            translation_cache.to_csv(translation_cache_path, index=False, encoding='utf-8')
            print(f'Progreso: {index}/{len(tags_to_translate)} tags traducidos y guardados.')
            pending_rows = []
if pending_rows:
    translation_cache = pd.concat([translation_cache, pd.DataFrame(pending_rows)], ignore_index=True)
    translation_cache = translation_cache.drop_duplicates(subset=['tag_clean'], keep='last').reset_index(drop=True)
    translation_cache.to_csv(translation_cache_path, index=False, encoding='utf-8')

translation_map = dict(zip(translation_cache['tag_clean'], translation_cache['tag_translated']))

tags_clean['tag_en'] = tags_clean['tag_clean'].map(translation_map).fillna(tags_clean['tag_clean'])
tags_clean['tag_en'] = tags_clean['tag_en'].map(normalize_text)
tags_clean = tags_clean[tags_clean['tag_en'].str.len() >= 3].copy()
tags_clean = tags_clean.drop_duplicates(subset=['movieId', 'tag_en']).reset_index(drop=True)

tags_grouped_en = (
    tags_clean.groupby('movieId')['tag_en']
    .apply(lambda values: ' '.join(sorted(set(tag for tag in values if tag))))
    .reset_index()
    .rename(columns={'tag_en': 'tags_combined_en'})
)

movies_with_tags = movies_clean.merge(tags_grouped_en, on='movieId', how='left')
movies_with_tags['tags_combined_en'] = movies_with_tags['tags_combined_en'].fillna('')
movies_with_tags = movies_with_tags.reset_index(drop=True)

print('Numero de tags originales (no nulos):', original_tag_count)
print('Numero de tags tras normalizar:', tags_after_normalize)
print('Numero de tags tras is_useful_tag:', tags_after_useful)
print('Numero de tags tras MIN_TAG_FREQUENCY:', tags_after_frequency)
print('Numero de tags unicos validos:', unique_valid_tags)
print('Numero de tags en cache:', len(cached_tags))
print('Numero de tags traducidos en esta ejecucion:', len(tags_to_translate))

# Guardado final completo del cache por seguridad.
translation_cache = translation_cache.drop_duplicates(subset=['tag_clean'], keep='last').reset_index(drop=True)
translation_cache.to_csv(translation_cache_path, index=False, encoding='utf-8')

display(translation_cache[['tag_clean', 'tag_translated', 'translation_changed']].head(10))
display(tags_clean[['tag_clean', 'tag_en']].drop_duplicates().head(10))
display(movies_with_tags[['movieId', 'title', 'genres', 'tags_combined_en']].head(10))

Numero total de tags unicos: 56351
Numero de tags ya en cache: 50
Numero de tags pendientes de traducir: 56301
Progreso: 10/56301 tags traducidos y guardados.
Progreso: 20/56301 tags traducidos y guardados.
Progreso: 30/56301 tags traducidos y guardados.
Progreso: 40/56301 tags traducidos y guardados.
Progreso: 50/56301 tags traducidos y guardados.
Progreso: 60/56301 tags traducidos y guardados.
Progreso: 70/56301 tags traducidos y guardados.
Progreso: 80/56301 tags traducidos y guardados.
Progreso: 90/56301 tags traducidos y guardados.
Progreso: 100/56301 tags traducidos y guardados.
Progreso: 110/56301 tags traducidos y guardados.
Progreso: 120/56301 tags traducidos y guardados.
Progreso: 130/56301 tags traducidos y guardados.
Progreso: 140/56301 tags traducidos y guardados.
Progreso: 150/56301 tags traducidos y guardados.
Progreso: 160/56301 tags traducidos y guardados.
Progreso: 170/56301 tags traducidos y guardados.
Progreso: 180/56301 tags traducidos y guardados.
Progreso: 190/56

KeyboardInterrupt: 

## 5. Creacion de content_features

Combinamos generos y `tags_combined_en` en una sola columna de texto.

Esta representacion es mas rica que usar solo generos, porque permite capturar similitudes mas especificas entre peliculas y mantiene el texto unificado en ingles.

In [ ]:
movies_with_tags['genres_text'] = movies_with_tags['genres'].fillna('').str.replace('|', ' ', regex=False)
movies_with_tags['content_features'] = (
    movies_with_tags['genres_text'].str.strip() + ' ' + movies_with_tags['tags_combined_en'].str.strip()
).str.strip()

display(movies_with_tags[['title', 'genres', 'tags_combined_en', 'content_features']].head())

## 6. Vectorizacion TF-IDF y similitud coseno

Usamos TF-IDF para convertir el texto en vectores numericos y calcular similitud coseno sobre demanda.

No construimos una matriz de similitud completa para todo el catalogo, porque seria muy costosa en memoria. En su lugar, calculamos las similitudes solo para la pelicula consultada.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words='english',
    min_df=2,
    max_df=0.85,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z0-9\-]{2,}\b',
)
content_matrix = tfidf_vectorizer.fit_transform(movies_with_tags['content_features'])

print('Forma de la matriz TF-IDF:', content_matrix.shape)
print('Numero de terminos:', len(tfidf_vectorizer.get_feature_names_out()))

## 7. Valoracion ponderada y popularidad suavizada

La valoracion ponderada corrige el sesgo de peliculas con pocas valoraciones.

Usamos la formula:

`weighted_rating = ((v / (v + m)) * R) + ((m / (v + m)) * C)`

donde:
- `v` es `rating_count`;
- `R` es `rating_mean`;
- `C` es la media global de `rating_mean`;
- `m` es el minimo de valoraciones usado como referencia.

La popularidad se calcula con `np.log1p(rating_count)` para suavizar diferencias muy grandes entre peliculas muy populares y peliculas poco conocidas.

In [ ]:
rating_reference_m = 20

movies_advanced = movies_with_tags.copy()
movies_advanced['rating_count'] = pd.to_numeric(movies_advanced['rating_count'], errors='coerce').fillna(0)
movies_advanced['rating_mean'] = pd.to_numeric(movies_advanced['rating_mean'], errors='coerce')
movies_advanced['year'] = pd.to_numeric(movies_advanced['year'], errors='coerce').astype('Int64')

global_mean_rating = movies_advanced['rating_mean'].mean()
v = movies_advanced['rating_count']
R = movies_advanced['rating_mean'].fillna(global_mean_rating)
C = global_mean_rating
m = rating_reference_m

movies_advanced['weighted_rating'] = ((v / (v + m)) * R) + ((m / (v + m)) * C)
movies_advanced['popularity_score'] = np.log1p(v)


def normalize_series(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors='coerce')
    min_value = numeric.min()
    max_value = numeric.max()

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.5, index=series.index)

    return (numeric - min_value) / (max_value - min_value)


movies_advanced['rating_score'] = normalize_series(movies_advanced['weighted_rating'])
movies_advanced['popularity_score'] = normalize_series(movies_advanced['popularity_score'])

display(movies_advanced[['title', 'rating_mean', 'rating_count', 'weighted_rating', 'rating_score', 'popularity_score']].head())

## 8. Funciones de recomendacion

La funcion principal devuelve recomendaciones ordenadas por `final_score`, que mezcla:

- similitud de contenido;
- puntuacion de valoracion;
- popularidad suavizada.

Tambien se incluyen filtros opcionales por ano y por generos incluidos o excluidos.

In [ ]:
def _empty_advanced_recommendations() -> pd.DataFrame:
    columns = [
        'title',
        'year',
        'genres',
        'rating_mean',
        'rating_count',
        'similarity_score',
        'weighted_rating',
        'rating_score',
        'popularity_score',
        'final_score',
    ]
    if 'title_clean' in movies_advanced.columns:
        columns.insert(1, 'title_clean')
    return pd.DataFrame(columns=columns)


def _normalize_genre_list(genres):
    if genres is None:
        return []
    if isinstance(genres, str):
        genres = [genres]
    return [str(genre).strip().lower() for genre in genres if str(genre).strip()]


def _split_movie_genres(genres_value):
    if pd.isna(genres_value):
        return set()
    return {genre.strip().lower() for genre in str(genres_value).split('|') if genre.strip()}


def _apply_genre_filters(df: pd.DataFrame, include_genres=None, exclude_genres=None) -> pd.DataFrame:
    result = df.copy()
    include_genres = _normalize_genre_list(include_genres)
    exclude_genres = _normalize_genre_list(exclude_genres)

    if include_genres:
        result = result[result['genres'].apply(lambda value: set(include_genres).issubset(_split_movie_genres(value)))]

    if exclude_genres:
        result = result[~result['genres'].apply(lambda value: bool(_split_movie_genres(value) & set(exclude_genres)))]

    return result


def _get_content_similarity_scores(movie_index: int) -> np.ndarray:
    return cosine_similarity(content_matrix[movie_index], content_matrix).flatten()


def _build_advanced_candidates(
    movie_title,
    min_ratings=20,
    min_year=None,
    max_year=None,
    include_genres=None,
    exclude_genres=None,
    content_weight=0.65,
    rating_weight=0.25,
    popularity_weight=0.10,
):
    selected_index = find_movie_index(movies_advanced, movie_title)
    if selected_index is None:
        return _empty_advanced_recommendations(), None

    candidate = movies_advanced.copy()
    similarity_scores = _get_content_similarity_scores(selected_index)
    candidate['similarity_score'] = similarity_scores
    candidate = candidate.drop(index=selected_index)

    if min_ratings is not None:
        candidate = candidate[candidate['rating_count'] >= min_ratings]

    if min_year is not None and 'year' in candidate.columns:
        candidate = candidate[candidate['year'].fillna(-1) >= min_year]

    if max_year is not None and 'year' in candidate.columns:
        candidate = candidate[candidate['year'].fillna(10**9) <= max_year]

    candidate = _apply_genre_filters(candidate, include_genres=include_genres, exclude_genres=exclude_genres)

    if candidate.empty:
        return _empty_advanced_recommendations(), selected_index

    weight_total = content_weight + rating_weight + popularity_weight
    if weight_total <= 0:
        raise ValueError('La suma de los pesos debe ser mayor que 0.')

    content_w = content_weight / weight_total
    rating_w = rating_weight / weight_total
    popularity_w = popularity_weight / weight_total

    candidate = candidate.copy()
    candidate['final_score'] = (
        content_w * candidate['similarity_score']
        + rating_w * candidate['rating_score']
        + popularity_w * candidate['popularity_score']
    )

    if 'title_clean' in candidate.columns:
        candidate['title'] = candidate['title_clean'].fillna(candidate['title'])

    columns = [
        'title',
        'year',
        'genres',
        'rating_mean',
        'rating_count',
        'similarity_score',
        'weighted_rating',
        'rating_score',
        'popularity_score',
        'final_score',
    ]
    if 'title_clean' in candidate.columns:
        columns.insert(1, 'title_clean')

    candidate = candidate.sort_values('final_score', ascending=False, kind='mergesort')
    candidate = candidate[columns].reset_index(drop=True)
    return candidate, selected_index


def recommend_movies_advanced(
    movie_title,
    n=10,
    min_ratings=20,
    min_year=None,
    max_year=None,
    include_genres=None,
    exclude_genres=None,
    content_weight=0.65,
    rating_weight=0.25,
    popularity_weight=0.10,
):
    candidates, selected_index = _build_advanced_candidates(
        movie_title=movie_title,
        min_ratings=min_ratings,
        min_year=min_year,
        max_year=max_year,
        include_genres=include_genres,
        exclude_genres=exclude_genres,
        content_weight=content_weight,
        rating_weight=rating_weight,
        popularity_weight=popularity_weight,
    )

    if selected_index is None:
        return candidates

    selected_title = movies_advanced.loc[selected_index, 'title']
    print(f'Pelicula seleccionada: {selected_title}')

    if candidates.empty:
        print('No se encontraron recomendaciones con los filtros indicados.')
        return candidates

    return candidates.head(n).reset_index(drop=True)


def recommend_movies_advanced_by_similarity(
    movie_title,
    n=10,
    min_ratings=20,
    min_year=None,
    max_year=None,
    include_genres=None,
    exclude_genres=None,
):
    candidates, selected_index = _build_advanced_candidates(
        movie_title=movie_title,
        min_ratings=min_ratings,
        min_year=min_year,
        max_year=max_year,
        include_genres=include_genres,
        exclude_genres=exclude_genres,
        content_weight=1.0,
        rating_weight=0.0,
        popularity_weight=0.0,
    )

    if selected_index is None or candidates.empty:
        return candidates

    selected_title = movies_advanced.loc[selected_index, 'title']
    print(f'Pelicula seleccionada: {selected_title}')
    return candidates.sort_values('similarity_score', ascending=False, kind='mergesort').head(n).reset_index(drop=True)


def recommend_movies_by_content_tags(movie_title, n=10, min_ratings=20):
    candidates, selected_index = _build_advanced_candidates(
        movie_title=movie_title,
        min_ratings=min_ratings,
        content_weight=1.0,
        rating_weight=0.0,
        popularity_weight=0.0,
    )

    if selected_index is None or candidates.empty:
        return candidates[[col for col in candidates.columns if col in ['title', 'year', 'genres', 'rating_mean', 'rating_count', 'similarity_score']]]

    selected_title = movies_advanced.loc[selected_index, 'title']
    print(f'Pelicula seleccionada: {selected_title}')

    output_columns = ['title', 'year', 'genres', 'rating_mean', 'rating_count', 'similarity_score']
    if 'title_clean' in candidates.columns:
        output_columns.insert(1, 'title_clean')

    return candidates.sort_values('similarity_score', ascending=False, kind='mergesort').head(n)[output_columns].reset_index(drop=True)


def compare_recommenders(movie_title, n=10, min_ratings=20, min_year=None, max_year=None, include_genres=None, exclude_genres=None):
    genre_columns = get_genre_columns(movies_clean)
    genre_feature_matrix = build_genre_feature_matrix(movies_clean, genre_columns)

    genre_only = recommend_movies_by_genres(movies_clean, genre_feature_matrix, movie_title, n=n, min_ratings=min_ratings)
    genre_tags = recommend_movies_by_content_tags(movie_title, n=n, min_ratings=min_ratings)
    advanced = recommend_movies_advanced(
        movie_title,
        n=n,
        min_ratings=min_ratings,
        min_year=min_year,
        max_year=max_year,
        include_genres=include_genres,
        exclude_genres=exclude_genres,
    )

    def _pad_titles(df):
        titles = df['title'].head(n).tolist()
        return titles + [None] * (n - len(titles))

    comparison = pd.DataFrame({
        'rank': range(1, n + 1),
        'generos': _pad_titles(genre_only),
        'generos_tags': _pad_titles(genre_tags),
        'avanzado': _pad_titles(advanced),
    })

    return genre_only, genre_tags, advanced, comparison

## 9. Ejemplos de uso

A continuacion se muestran varios ejemplos para comprobar el comportamiento del recomendador avanzado con distintos filtros.

In [ ]:
toy_story_recs = recommend_movies_advanced('Interstellar', n=10)
display(toy_story_recs)

In [ ]:
matrix_recs = recommend_movies_advanced('The Matrix', n=10)
display(matrix_recs)

In [ ]:
matrix_recent_recs = recommend_movies_advanced('The Matrix', n=10, min_year=2000)
display(matrix_recent_recs)

In [ ]:
matrix_no_horror_recs = recommend_movies_advanced('The Matrix', n=10, exclude_genres=['Horror'])
display(matrix_no_horror_recs)

In [ ]:
toy_story_scifi_recs = recommend_movies_advanced('Toy Story', n=10, include_genres=['Sci-Fi'])
display(toy_story_scifi_recs)

## 10. Comparacion entre ordenacion por similitud y por final_score

Esta comparacion ayuda a ver la diferencia entre un criterio puramente textual y un criterio mixto que tambien considera valoracion y popularidad.

In [ ]:
matrix_similarity_only = recommend_movies_advanced_by_similarity('The Matrix', n=10)
matrix_final_score = recommend_movies_advanced('The Matrix', n=10)

print('Ordenado solo por similarity_score')
display(matrix_similarity_only)

print('Ordenado por final_score')
display(matrix_final_score)

## 11. Comparacion de los tres recomendadores

Aqui se comparan de forma directa:
- el recomendador basado solo en generos;
- el recomendador basado en generos + tags;
- el recomendador avanzado que mezcla similitud, valoracion y popularidad.

La comparacion es util para la memoria del TFG porque permite justificar por que la version avanzada produce resultados mas equilibrados.

In [ ]:
genre_only, genre_tags, advanced, comparison_toy_story = compare_recommenders('28 days later', n=10)

print('Recomendador solo con generos')
display(genre_only)

print('Recomendador con generos + tags')
display(genre_tags)

print('Recomendador avanzado')
display(advanced)

print('Comparacion compacta por rango')
display(comparison_toy_story)

In [ ]:
genre_only_matrix, genre_tags_matrix, advanced_matrix, comparison_matrix = compare_recommenders('The Matrix', n=10)

print('Recomendador solo con generos')
display(genre_only_matrix)

print('Recomendador con generos + tags')
display(genre_tags_matrix)

print('Recomendador avanzado')
display(advanced_matrix)

print('Comparacion compacta por rango')
display(comparison_matrix)

## 12. Ideas para explicar en la memoria

- Los tags mejoran los generos porque capturan detalles mas especificos que no aparecen en la codificacion binaria de generos.
- `weighted_rating` evita que una pelicula con muy pocos votos y una media alta aparezca artificialmente arriba.
- `popularity_score` ayuda a priorizar peliculas con mas base de valoraciones, lo que reduce ruido.
- La combinacion de factores hace que el recomendador sea mas completo porque no depende de una sola senal.

Con esta version ya queda preparado un recomendador mas defendible para el TFG, sin usar APIs externas ni tecnicas de deep learning.